This file gathers the weather data for the airports for the year 2024, and stores the data into csv files that can be read again later


In [12]:
import os.path

import boto3
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config
import io

In [8]:
# Setup Boto3 client for S3 with unsigned configuration
s3_client = boto3.client('s3', config=Config(signature_version=UNSIGNED))
bucket_name = 'noaa-global-hourly-pds'

In [9]:
output_dir = '2024_weather'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [10]:
def download_airport_weather_yearly(station_id, year, airport_id):
    """
    Fetches hourly weather data for a specific station and filters by date.
    :param station_id: The ICAO station ID for the weather station
    :param year: The year for which to fetch the weather data
    :param airport_id: The corresponding airport ID (for naming the output file)
    """
    # Construct the exact S3 Key based on the document's path pattern
    file_key = f"{year}/{station_id}.csv"

    try:
        print(f"Fetching data from: s3://{bucket_name}/{file_key}")

        # Get the object directly (No need to list/paginate)
        response = s3_client.get_object(Bucket=bucket_name, Key=file_key)

        # Read the raw comma-separated bytes directly into a Pandas DataFrame
        df = pd.read_csv(io.BytesIO(response['Body'].read()), low_memory=False)

        # Save the dataframe as a CSV file to the output directory
        output_path = os.path.join(output_dir, f'airport_id_{airport_id}_year_{year}')
        df.to_csv(output_path)
        print(f"Saved weather data for station {station_id} (airport ID {airport_id}) for year {year} to {output_path}")

    except Exception as e:
        print(f"Error retrieving data: {e}")
        return None

In [11]:
# Load the airports that we need to gather data for
airports_df = pd.read_csv('top_100.csv')

# Create the station IDs to get the weather data for
for index, row in airports_df.iterrows():
    usaf = row['USAF']
    wban = row['WBAN']
    airport_id = row['AIRPORT_ID']

    # correct usaf and wban to be zero-padded strings of the correct length
    usaf_str = str(int(usaf)).zfill(6)
    wban_str = str(int(wban)).zfill(5)
    station_id = usaf_str + wban_str
    download_airport_weather_yearly(station_id, 2024, airport_id)

Fetching data from: s3://noaa-global-hourly-pds/2024/72219013874.csv
Saved weather data for station 72219013874 (airport ID 10397) for year 2024 to 2024_weather\airport_id_10397_year_2024
Fetching data from: s3://noaa-global-hourly-pds/2024/72530094846.csv
Saved weather data for station 72530094846 (airport ID 13930) for year 2024 to 2024_weather\airport_id_13930_year_2024
Fetching data from: s3://noaa-global-hourly-pds/2024/72259003927.csv
Saved weather data for station 72259003927 (airport ID 11298) for year 2024 to 2024_weather\airport_id_11298_year_2024
Fetching data from: s3://noaa-global-hourly-pds/2024/72467003017.csv
Error retrieving data: An error occurred (NoSuchKey) when calling the GetObject operation: The specified key does not exist.
Fetching data from: s3://noaa-global-hourly-pds/2024/72565003017.csv
Saved weather data for station 72565003017 (airport ID 11292) for year 2024 to 2024_weather\airport_id_11292_year_2024
Fetching data from: s3://noaa-global-hourly-pds/2024/7

## Using weather data
The below code shows how we can map airport IDs and timestamps to weather

In [22]:
def load_all_weather_data(weather_dir):
    airports_df = pd.read_csv('top_100.csv')
    weather_data = {}

    # Load in every weather csv file based on its airport id
    # Get all the files in the weather directory
    for filename in os.listdir(weather_dir):
        if filename.endswith(".csv"):
            file_path = os.path.join(weather_dir, filename)
            # Extract the airport ID from the filename
            airport_id = filename.split('_')[2]  # Assuming the format is 'airport_id_{airport_id}_year_{year}.csv'

            # Read the weather data for this airport
            df = pd.read_csv(file_path, low_memory=False)
            # Convert the DATE column to datetime objects
            df['DATE'] = pd.to_datetime(df['DATE'])
            # Store the dataframe in the dictionary
            weather_data[airport_id] = df

    return weather_data

In [23]:
weather_dir = '2024_weather'
weather_data = load_all_weather_data(weather_dir)

In [29]:
def get_weather_for_flight(airport_id, timestamp, weather_data):
    """
    Retrieves the weather data for a specific airport and timestamp.
    :param airport_id: The ID of the airport
    :param timestamp: The timestamp for which to retrieve the weather data (in datetime format)
    :param weather_data: A dictionary containing the weather dataframes for each airport
    :return: A dictionary of weather conditions for the given airport and timestamp
    """
    if airport_id not in weather_data:
        print(f"No weather data available for airport ID {airport_id}")
        return None

    df = weather_data[airport_id]

    # searchsorted finds the exact index where the timestamp
    # should be inserted to maintain the sorted order.
    idx = df['DATE'].searchsorted(timestamp, side='right')

    # If idx is 0, it means the timestamp is earlier than the very first record.
    if idx == 0:
        print(f"Warning: No weather data found on or before {timestamp} for airport {airport_id}.")
        return None

    # The closest row before or equal to the timestamp is exactly one index prior.
    closest_row = df.iloc[idx - 1]

    return closest_row

In [30]:
# Test the weather retrieval function with a sample airport ID and timestamp
airport_id = '15023'
timestamp = pd.to_datetime('2024-01-15 12:00:00')
weather_conditions = get_weather_for_flight(airport_id, timestamp, weather_data)
print(weather_conditions)

Unnamed: 0                                                  581
STATION                                             72495723213
DATE                                        2024-01-15 11:53:00
SOURCE                                                        7
LATITUDE                                               38.50369
                                    ...                        
RH1                                                         NaN
RH2                                                         NaN
RH3                                                         NaN
REM           MET11001/15/24 03:53:02 METAR KSTS 151153Z 000...
EQD                                                         NaN
Name: 581, Length: 88, dtype: object
